# OLMoE Architecture Inspection
**Task 1 — Generalization Specialist**

Before writing any adapter code, we need to answer four questions:
1. Are expert weights packed into one tensor (like Phi), or stored as separate modules?
2. What is the calling convention — how does the MoE block call its experts?
3. What are the exact dimensions (model_dim, ffn_dim)?
4. Does `output_router_logits=True` work, and what shape does it return?

These answers determine how much of `HierarchicalPhimoeExperts` can be reused vs. rewritten.

**Phi-3.5-MoE reference (what we already know works):**
- 32 layers, 16 experts/layer, top-2 routing
- Experts packed: `down_proj.shape = [16, 4096, 6400]`
- Calling convention: `experts(hidden_states, top_k_ids, top_k_weights)`
- LoRA dimension: `model_dim = 4096`
- Critical bug to avoid: using `ffn_dim` instead of `model_dim` for LoRA

**OLMoE target:**
- Expected: 16 layers, 64 experts/layer, top-8 routing
- Everything else is unknown until we inspect it below

## Cell 1 — Install & Load Model

In [4]:
!pip install transformers accelerate -q

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "allenai/OLMoE-1B-7B-0125"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

print(f"Model class: {type(model).__name__}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

Model class: OlmoeForCausalLM
VRAM used: 13.85 GB


## Cell 2 — Smoke Test
Verify the model loads and generates correctly before touching internals.

In [6]:
prompt = "Explain in simple terms what a Mixture of Experts model is."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True
    )

response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

Explain in simple terms what a Mixture of Experts model is.

A mixture of experts is a method to combine the outputs of several experts. Each expert in the mixture of experts is a function that maps a set of input variables to a probability distribution over a set of possible outputs. The mixture of experts model attempts to combine these probability distributions to produce a single output distribution.

What is the difference between a mixture of experts and a regular ensemble?

A mixture of experts model is an ensemble of experts. The difference is that a regular ensemble produces


## Cell 3 — Top-Level Architecture
Print the high-level module tree. This tells us the naming convention
for layers, MoE blocks, and experts before we dig deeper.

In [7]:
print("=" * 60)
print("TOP-LEVEL MODULE STRUCTURE")
print("=" * 60)
for name, module in model.named_children():
    print(f"  {name}: {type(module).__name__}")

print()
print("=" * 60)
print("DEPTH-2 MODULES")
print("=" * 60)
for name, module in model.named_modules():
    depth = name.count('.')
    if depth == 2:
        print(f"  {name}: {type(module).__name__}")

TOP-LEVEL MODULE STRUCTURE
  model: OlmoeModel
  lm_head: Linear

DEPTH-2 MODULES
  model.layers.0: OlmoeDecoderLayer
  model.layers.1: OlmoeDecoderLayer
  model.layers.2: OlmoeDecoderLayer
  model.layers.3: OlmoeDecoderLayer
  model.layers.4: OlmoeDecoderLayer
  model.layers.5: OlmoeDecoderLayer
  model.layers.6: OlmoeDecoderLayer
  model.layers.7: OlmoeDecoderLayer
  model.layers.8: OlmoeDecoderLayer
  model.layers.9: OlmoeDecoderLayer
  model.layers.10: OlmoeDecoderLayer
  model.layers.11: OlmoeDecoderLayer
  model.layers.12: OlmoeDecoderLayer
  model.layers.13: OlmoeDecoderLayer
  model.layers.14: OlmoeDecoderLayer
  model.layers.15: OlmoeDecoderLayer


## Cell 4 — Find MoE Layers
Locate every module with 'expert', 'moe', or 'sparse' in its name or class.
This gives us the exact path to patch.

In [8]:
print("=" * 60)
print("ALL MoE-RELATED MODULES")
print("=" * 60)

moe_modules = []
for name, module in model.named_modules():
    class_name = type(module).__name__.lower()
    name_lower  = name.lower()
    if any(kw in class_name or kw in name_lower
           for kw in ['expert', 'moe', 'sparse', 'router', 'gate']):
        depth = name.count('.')
        if depth <= 5:   # skip deeply nested noise
            print(f"  [depth {depth}] {name}: {type(module).__name__}")
            moe_modules.append((name, module))

print(f"\nTotal MoE-related modules found: {len(moe_modules)}")

ALL MoE-RELATED MODULES
  [depth 0] : OlmoeForCausalLM
  [depth 0] model: OlmoeModel
  [depth 2] model.layers.0: OlmoeDecoderLayer
  [depth 3] model.layers.0.self_attn: OlmoeAttention
  [depth 4] model.layers.0.self_attn.q_norm: OlmoeRMSNorm
  [depth 4] model.layers.0.self_attn.k_norm: OlmoeRMSNorm
  [depth 3] model.layers.0.mlp: OlmoeSparseMoeBlock
  [depth 4] model.layers.0.mlp.gate: OlmoeTopKRouter
  [depth 4] model.layers.0.mlp.experts: OlmoeExperts
  [depth 5] model.layers.0.mlp.experts.act_fn: SiLUActivation
  [depth 3] model.layers.0.input_layernorm: OlmoeRMSNorm
  [depth 3] model.layers.0.post_attention_layernorm: OlmoeRMSNorm
  [depth 2] model.layers.1: OlmoeDecoderLayer
  [depth 3] model.layers.1.self_attn: OlmoeAttention
  [depth 4] model.layers.1.self_attn.q_norm: OlmoeRMSNorm
  [depth 4] model.layers.1.self_attn.k_norm: OlmoeRMSNorm
  [depth 3] model.layers.1.mlp: OlmoeSparseMoeBlock
  [depth 4] model.layers.1.mlp.gate: OlmoeTopKRouter
  [depth 4] model.layers.1.mlp.expert

## Cell 5 — Inspect One Full Layer
Print the complete structure of layer 0, including all parameter shapes.
This is where we see if experts are packed or separate.

In [9]:
print("=" * 60)
print("FULL STRUCTURE OF LAYER 0")
print("=" * 60)

# Try common path patterns
layer0 = None
for attr in ['model', 'transformer']:
    if hasattr(model, attr):
        inner = getattr(model, attr)
        if hasattr(inner, 'layers'):
            layer0 = inner.layers[0]
            print(f"Path: model.{attr}.layers[0]")
            break
        elif hasattr(inner, 'blocks'):
            layer0 = inner.blocks[0]
            print(f"Path: model.{attr}.blocks[0]")
            break

if layer0 is None:
    raise RuntimeError("Could not find layer 0 — inspect model structure above and update path")

print()
for name, module in layer0.named_modules():
    if name == '':
        continue
    depth = name.count('.')
    indent = '  ' * depth
    print(f"{indent}{name}: {type(module).__name__}")

print()
print("PARAMETERS IN LAYER 0:")
for name, param in layer0.named_parameters():
    print(f"  {name}: {param.shape} dtype={param.dtype}")

FULL STRUCTURE OF LAYER 0
Path: model.model.layers[0]

self_attn: OlmoeAttention
  self_attn.q_proj: Linear
  self_attn.k_proj: Linear
  self_attn.v_proj: Linear
  self_attn.o_proj: Linear
  self_attn.q_norm: OlmoeRMSNorm
  self_attn.k_norm: OlmoeRMSNorm
mlp: OlmoeSparseMoeBlock
  mlp.gate: OlmoeTopKRouter
  mlp.experts: OlmoeExperts
    mlp.experts.act_fn: SiLUActivation
input_layernorm: OlmoeRMSNorm
post_attention_layernorm: OlmoeRMSNorm

PARAMETERS IN LAYER 0:
  self_attn.q_proj.weight: torch.Size([2048, 2048]) dtype=torch.float16
  self_attn.k_proj.weight: torch.Size([2048, 2048]) dtype=torch.float16
  self_attn.v_proj.weight: torch.Size([2048, 2048]) dtype=torch.float16
  self_attn.o_proj.weight: torch.Size([2048, 2048]) dtype=torch.float16
  self_attn.q_norm.weight: torch.Size([2048]) dtype=torch.float16
  self_attn.k_norm.weight: torch.Size([2048]) dtype=torch.float16
  mlp.gate.weight: torch.Size([64, 2048]) dtype=torch.float16
  mlp.experts.gate_up_proj: torch.Size([64, 2048, 

## Cell 6 — CRITICAL: Packed vs Separate Experts

**This is the most important cell.**

Phi-3.5-MoE packs all 16 experts into a single tensor:
```
experts.down_proj.shape = [16, 4096, 6400]   ← one tensor for all experts
```

OLMoE might store each expert separately:
```
experts[0].down_proj.shape = [4096, 6400]    ← one module per expert
experts[1].down_proj.shape = [4096, 6400]
...
```

The answer here determines whether `HierarchicalPhimoeExperts` can be adapted
or needs to be rewritten from scratch.

In [10]:
print("=" * 60)
print("PACKED VS SEPARATE EXPERT CHECK")
print("=" * 60)

# Navigate to the MoE FFN block — update path based on Cell 4 findings
# Common OLMoE paths to try:
candidates = [
    lambda: model.model.layers[0].mlp,
    lambda: model.model.layers[0].block_sparse_moe,
    lambda: model.model.layers[0].moe,
    lambda: model.model.layers[0].ffn,
]

moe_block = None
for fn in candidates:
    try:
        moe_block = fn()
        print(f"Found MoE block: {type(moe_block).__name__}")
        break
    except AttributeError:
        continue

if moe_block is None:
    print("Could not auto-find MoE block — use the path from Cell 4 above")
else:
    # Check for expert container
    expert_container = None
    for attr in ['experts', 'ffn', 'mlp']:
        if hasattr(moe_block, attr):
            expert_container = getattr(moe_block, attr)
            print(f"Expert container attribute: .{attr}")
            print(f"Expert container type: {type(expert_container).__name__}")
            break

    if expert_container is not None:
        print()
        # Case 1: nn.ModuleList (separate modules per expert)
        if isinstance(expert_container, torch.nn.ModuleList):
            print("RESULT: SEPARATE modules (nn.ModuleList)")
            print(f"  Number of experts: {len(expert_container)}")
            print(f"  Expert[0] type: {type(expert_container[0]).__name__}")
            print(f"  Expert[0] parameters:")
            for pname, param in expert_container[0].named_parameters():
                print(f"    {pname}: {param.shape}")

        # Case 2: packed tensor (like Phi)
        elif hasattr(expert_container, 'down_proj'):
            dp = expert_container.down_proj
            if hasattr(dp, 'shape') and len(dp.shape) == 3:
                print("RESULT: PACKED tensor (like Phi-MoE)")
                print(f"  down_proj.shape: {dp.shape}")
                print(f"  → [num_experts={dp.shape[0]}, out_f={dp.shape[1]}, in_f={dp.shape[2]}]")
            else:
                print(f"RESULT: down_proj exists but shape is {dp.shape} — inspect manually")
        else:
            print("Could not determine packing — printing all attributes:")
            for attr_name in dir(expert_container):
                if not attr_name.startswith('_'):
                    val = getattr(expert_container, attr_name, None)
                    if isinstance(val, (torch.Tensor, torch.nn.Module, torch.nn.ModuleList)):
                        print(f"  .{attr_name}: {type(val).__name__}")
                        if isinstance(val, torch.Tensor):
                            print(f"    shape: {val.shape}")

PACKED VS SEPARATE EXPERT CHECK
Found MoE block: OlmoeSparseMoeBlock
Expert container attribute: .experts
Expert container type: OlmoeExperts

RESULT: PACKED tensor (like Phi-MoE)
  down_proj.shape: torch.Size([64, 2048, 1024])
  → [num_experts=64, out_f=2048, in_f=1024]


## Cell 7 — Extract Key Dimensions

Find `model_dim`, `ffn_dim`, number of experts, and top-k.

In Phi the critical bug was using `ffn_dim=6400` instead of `model_dim=4096`
for the LoRA dimension. We need to know OLMoE's values before writing any LoRA code.

In [11]:
print("=" * 60)
print("KEY DIMENSIONS")
print("=" * 60)

cfg = model.config
print(f"Config class: {type(cfg).__name__}")
print()

# Print all config fields that relate to dimensions or routing
dimension_keys = [
    'hidden_size', 'intermediate_size', 'num_hidden_layers',
    'num_attention_heads', 'num_key_value_heads',
    'num_experts', 'num_experts_per_tok', 'num_local_experts',
    'top_k', 'router_top_k', 'moe_top_k',
    'ffn_dim', 'model_dim',
    'expert_intermediate_size', 'moe_intermediate_size',
]

found = {}
for key in dimension_keys:
    if hasattr(cfg, key):
        val = getattr(cfg, key)
        print(f"  config.{key} = {val}")
        found[key] = val

# Also print anything we missed that looks dimension-like
print()
print("Other config fields (scanning for anything missed):")
for key, val in vars(cfg).items():
    if key not in dimension_keys and isinstance(val, int) and 10 <= val <= 100000:
        print(f"  config.{key} = {val}")

print()
print("=" * 60)
print("INTERPRETATION:")
model_dim = found.get('hidden_size', '???')
ffn_dim   = found.get('intermediate_size') or found.get('expert_intermediate_size', '???')
n_experts = found.get('num_experts') or found.get('num_local_experts', '???')
top_k     = found.get('num_experts_per_tok') or found.get('top_k') or found.get('router_top_k', '???')
print(f"  model_dim (LoRA dimension) = {model_dim}")
print(f"  ffn_dim  (expert internal) = {ffn_dim}")
print(f"  num_experts per layer      = {n_experts}")
print(f"  top_k routing              = {top_k}")
print()
print("CRITICAL: LoRA correction must live in model_dim → model_dim space")
print(f"          i.e. lora_in = lora_out = {model_dim}")
print(f"          NOT ffn_dim ({ffn_dim}) — that was the Phi bug")

KEY DIMENSIONS
Config class: OlmoeConfig

  config.hidden_size = 2048
  config.intermediate_size = 1024
  config.num_hidden_layers = 16
  config.num_attention_heads = 16
  config.num_key_value_heads = 16
  config.num_experts = 64
  config.num_experts_per_tok = 8
  config.num_local_experts = 64

Other config fields (scanning for anything missed):
  config.vocab_size = 50304
  config.max_position_embeddings = 4096
  config.eos_token_id = 50279

INTERPRETATION:
  model_dim (LoRA dimension) = 2048
  ffn_dim  (expert internal) = 1024
  num_experts per layer      = 64
  top_k routing              = 8

CRITICAL: LoRA correction must live in model_dim → model_dim space
          i.e. lora_in = lora_out = 2048
          NOT ffn_dim (1024) — that was the Phi bug


## Cell 8 — Router Structure

Find the router module and understand how it selects experts.
We need to know: does it use softmax + topk, or something else?

In [12]:
print("=" * 60)
print("ROUTER STRUCTURE")
print("=" * 60)

for name, module in model.named_modules():
    class_name = type(module).__name__.lower()
    if 'router' in class_name or 'gate' in name.lower() and 'layer' in name.lower():
        depth = name.count('.')
        if depth <= 5:
            print(f"\nFound: {name}")
            print(f"  Type: {type(module).__name__}")
            for pname, param in module.named_parameters():
                print(f"  {pname}: {param.shape}")
            # Try to print source
            import inspect
            try:
                src = inspect.getsource(type(module).forward)
                print(f"  forward() source:")
                for line in src.split('\n')[:30]:  # first 30 lines
                    print(f"    {line}")
            except Exception:
                pass
            break

ROUTER STRUCTURE

Found: model.layers.0.mlp.gate
  Type: OlmoeTopKRouter
  weight: torch.Size([64, 2048])
  forward() source:
        def forward(self, hidden_states):
            hidden_states = hidden_states.reshape(-1, self.hidden_dim)
            router_logits = F.linear(hidden_states, self.weight)  # (seq_len, num_experts)
            router_logits = torch.nn.functional.softmax(router_logits, dtype=torch.float, dim=-1)
            router_top_value, router_indices = torch.topk(router_logits, self.top_k, dim=-1)  # (seq_len, top_k)
            if self.norm_topk_prob:
                router_top_value /= router_top_value.sum(dim=-1, keepdim=True)
            router_top_value = router_top_value.to(router_logits.dtype)
            router_scores = router_top_value
            return router_logits, router_scores, router_indices
    


## Cell 9 — SparseMoeBlock Calling Convention

The most critical question: how does the MoE block call its expert container?

Phi does:
```python
output = self.experts(hidden_states, top_k_ids, top_k_weights)
```

OLMoE might loop through each expert individually, or use a different signature.
We inspect the source of the MoE block's forward() to find out.

In [13]:
import inspect

print("=" * 60)
print("MoE BLOCK forward() SOURCE")
print("=" * 60)

printed = set()
for name, module in model.named_modules():
    class_name = type(module).__name__
    if class_name in printed:
        continue
    if any(kw in class_name.lower() for kw in ['moe', 'sparse', 'mixture']):
        printed.add(class_name)
        print(f"\nClass: {class_name} (at path: {name})")
        print("-" * 50)
        try:
            src = inspect.getsource(type(module).forward)
            print(src)
        except Exception as e:
            print(f"Could not get source: {e}")
            print(f"Module repr: {module}")

MoE BLOCK forward() SOURCE

Class: OlmoeForCausalLM (at path: )
--------------------------------------------------
    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.LongTensor | None = None,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        inputs_embeds: torch.FloatTensor | None = None,
        labels: torch.LongTensor | None = None,
        use_cache: bool | None = None,
        output_router_logits: bool | None = None,
        cache_position: torch.LongTensor | None = None,
        logits_to_keep: int | torch.Tensor = 0,
        **kwargs: Unpack[TransformersKwargs],
    ) -> MoeCausalLMOutputWithPast:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the masked language modeling loss. Indices should either be in `[0, ...,
            config.vo

## Cell 10 — Test output_router_logits

The Jaccard diagnostic and `ConflictSaturationMonitor` both depend on being able to
extract per-layer routing decisions by passing `output_router_logits=True`.

Verify it works and check the shape of what comes back.

In [14]:
print("=" * 60)
print("output_router_logits TEST")
print("=" * 60)

test_input = tokenizer("The patient presented with fever.",
                        return_tensors="pt").to(model.device)

with torch.no_grad():
    try:
        out = model(**test_input, output_router_logits=True, return_dict=True)
        print("✅ output_router_logits=True is supported")
        print()
        if hasattr(out, 'router_logits') and out.router_logits is not None:
            logits_list = out.router_logits
            print(f"router_logits type: {type(logits_list).__name__}")
            print(f"Number of elements (should = num_layers): {len(logits_list)}")
            print()
            for i, rl in enumerate(logits_list[:3]):  # show first 3 layers
                print(f"  Layer {i}: shape={rl.shape}, dtype={rl.dtype}")
            print("  ...")
            print()
            rl0 = logits_list[0]
            print("INTERPRETATION of layer 0 router_logits:")
            if rl0.dim() == 2:
                T, E = rl0.shape
                print(f"  Shape: [{T} tokens, {E} experts]")
                print(f"  → Matches OLMoE expected (batch*seq, n_experts)")
            elif rl0.dim() == 3:
                B, S, E = rl0.shape
                print(f"  Shape: [{B} batch, {S} seq, {E} experts]")
                print(f"  → Needs .view(-1, {E}) before topk — same fix as Phi")
        else:
            print("❌ router_logits attribute is None or missing")
            print(f"   Available output keys: {list(out.keys()) if hasattr(out, 'keys') else dir(out)}")
    except Exception as e:
        print(f"❌ output_router_logits=True raised an error: {e}")
        print("   Will need a forward hook approach instead")

output_router_logits TEST
✅ output_router_logits=True is supported

router_logits type: tuple
Number of elements (should = num_layers): 16

  Layer 0: shape=torch.Size([6, 64]), dtype=torch.float32
  Layer 1: shape=torch.Size([6, 64]), dtype=torch.float32
  Layer 2: shape=torch.Size([6, 64]), dtype=torch.float32
  ...

INTERPRETATION of layer 0 router_logits:
  Shape: [6 tokens, 64 experts]
  → Matches OLMoE expected (batch*seq, n_experts)


## Cell 11 — Forward Hook Fallback (if Cell 10 fails)

If `output_router_logits` is not supported, we can intercept router outputs
using PyTorch forward hooks. This cell demonstrates the approach.

In [15]:
print("=" * 60)
print("FORWARD HOOK FALLBACK — captures routing without model support")
print("=" * 60)

captured_router_outputs = {}
hooks = []

# Register hooks on every router/gate module
for name, module in model.named_modules():
    class_name = type(module).__name__.lower()
    if 'router' in class_name or ('gate' in name and 'layer' in name):
        def make_hook(n):
            def hook(mod, inp, out):
                # out might be (routing_weights, selected_experts) tuple, or just logits
                captured_router_outputs[n] = out
            return hook
        h = module.register_forward_hook(make_hook(name))
        hooks.append(h)

test_input = tokenizer("The patient presented with fever.",
                        return_tensors="pt").to(model.device)

with torch.no_grad():
    _ = model(**test_input)

for h in hooks:
    h.remove()

print(f"Captured outputs from {len(captured_router_outputs)} router modules")
print()
for name, out in list(captured_router_outputs.items())[:3]:
    print(f"Router: {name}")
    if isinstance(out, tuple):
        print(f"  Output is a tuple of {len(out)} elements:")
        for i, elem in enumerate(out):
            if isinstance(elem, torch.Tensor):
                print(f"    [{i}]: shape={elem.shape}, dtype={elem.dtype}")
    elif isinstance(out, torch.Tensor):
        print(f"  Output tensor: shape={out.shape}, dtype={out.dtype}")
    print()

FORWARD HOOK FALLBACK — captures routing without model support
Captured outputs from 16 router modules

Router: model.layers.0.mlp.gate
  Output is a tuple of 3 elements:
    [0]: shape=torch.Size([6, 64]), dtype=torch.float32
    [1]: shape=torch.Size([6, 8]), dtype=torch.float32
    [2]: shape=torch.Size([6, 8]), dtype=torch.int64

Router: model.layers.1.mlp.gate
  Output is a tuple of 3 elements:
    [0]: shape=torch.Size([6, 64]), dtype=torch.float32
    [1]: shape=torch.Size([6, 8]), dtype=torch.float32
    [2]: shape=torch.Size([6, 8]), dtype=torch.int64

Router: model.layers.2.mlp.gate
  Output is a tuple of 3 elements:
    [0]: shape=torch.Size([6, 64]), dtype=torch.float32
    [1]: shape=torch.Size([6, 8]), dtype=torch.float32
    [2]: shape=torch.Size([6, 8]), dtype=torch.int64



## Cell 12 — Verify Zero-Output LoRA Patch

Before writing the full `HierarchicalOLMoEExperts` wrapper, verify the core guarantee:
a LoRA adapter with `B=0` produces zero output — meaning patching doesn't change the
model's behavior at initialization.

Run a forward pass before and after a manual patch, confirm the diff is ~0.

In [16]:
import torch.nn as nn

print("=" * 60)
print("ZERO-OUTPUT LoRA PATCH VERIFICATION")
print("=" * 60)

# Navigate to layer 0's MoE block — update path from Cell 5 findings
# We'll capture hidden states at that layer before and after

captured = {}

def make_capture_hook(label):
    def hook(mod, inp, out):
        captured[label] = out[0].detach().clone() if isinstance(out, tuple) else out.detach().clone()
    return hook

# Find the MoE FFN block in layer 0 and register a hook
moe_layer = None
for attr in ['mlp', 'block_sparse_moe', 'moe', 'ffn']:
    if hasattr(model.model.layers[0], attr):
        moe_layer = getattr(model.model.layers[0], attr)
        print(f"Hooking onto: model.model.layers[0].{attr}")
        break

if moe_layer is not None:
    test_input = tokenizer("Hello world", return_tensors="pt").to(model.device)

    # BEFORE patch
    hook = moe_layer.register_forward_hook(make_capture_hook('before'))
    with torch.no_grad():
        _ = model(**test_input)
    hook.remove()

    # Minimal inline LoRA: B=0 means output is unchanged
    model_dim = model.config.hidden_size
    rank = 16

    lora_A = nn.Parameter(torch.randn(rank, model_dim, device=model.device,
                                       dtype=torch.float16) * 0.01)
    lora_B = nn.Parameter(torch.zeros(model_dim, rank, device=model.device,
                                       dtype=torch.float16))  # B=0 → zero contribution

    # Manually compute lora output — should be exactly zero
    dummy_input = torch.randn(5, model_dim, device=model.device, dtype=torch.float16)
    lora_out = (dummy_input @ lora_A.T) @ lora_B.T

    print()
    print(f"LoRA output with B=0:")
    print(f"  max absolute value: {lora_out.abs().max().item():.8f}")
    print(f"  mean absolute value: {lora_out.abs().mean().item():.8f}")
    if lora_out.abs().max().item() < 1e-6:
        print("✅ Zero-output guarantee confirmed — B=0 produces exactly zero")
    else:
        print("❌ Unexpected non-zero output — check initialization")
else:
    print("Could not find MoE layer — update path from Cell 5")

ZERO-OUTPUT LoRA PATCH VERIFICATION
Hooking onto: model.model.layers[0].mlp

LoRA output with B=0:
  max absolute value: 0.00000000
  mean absolute value: 0.00000000
✅ Zero-output guarantee confirmed — B=0 produces exactly zero


## Cell 13 — Accelerate Hook Check

When Phi was patched, the accelerate `AlignDevicesHook` had to be manually
transferred from the original expert block to the wrapper. Forgetting this
caused device/dtype corruption in all subsequent layers.

Check whether OLMoE's expert block has this hook, and what needs to be transferred.

In [17]:
print("=" * 60)
print("ACCELERATE HOOK CHECK")
print("=" * 60)

hook_found = False
for name, module in model.named_modules():
    if hasattr(module, '_hf_hook'):
        print(f"Found _hf_hook on: {name}")
        print(f"  Hook type: {type(module._hf_hook).__name__}")
        hook_found = True

if not hook_found:
    print("No _hf_hook found on any module.")
    print("→ This is the single-GPU case — no hook transfer needed when patching.")
    print("→ If running on multi-GPU (device_map='auto' with 2 GPUs), re-run this")
    print("  check after loading and look for AlignDevicesHook.")
else:
    print()
    print("ACTION REQUIRED when patching:")
    print("  from accelerate.hooks import remove_hook_from_module, add_hook_to_module")
    print("  hook = original_experts._hf_hook")
    print("  remove_hook_from_module(original_experts)")
    print("  add_hook_to_module(your_wrapper, hook)")

ACCELERATE HOOK CHECK
Found _hf_hook on: 
  Hook type: AlignDevicesHook
Found _hf_hook on: model.embed_tokens
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.self_attn
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.self_attn.q_proj
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.self_attn.k_proj
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.self_attn.v_proj
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.self_attn.o_proj
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.self_attn.q_norm
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.self_attn.k_norm
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.mlp
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.mlp.gate
  Hook type: AlignDevicesHook
Found _hf_hook on: model.layers.0.mlp.experts
  Hook type: AlignDevicesHook
Found _hf_hook on: 

## Cell 14 — Summary and Comparison Table

Collect all findings and compare with Phi-3.5-MoE.
This tells us exactly what needs to change in `HierarchicalPhimoeExperts`.

In [18]:
cfg = model.config

model_dim = getattr(cfg, 'hidden_size', '???')
ffn_dim   = getattr(cfg, 'intermediate_size', None) or getattr(cfg, 'expert_intermediate_size', '???')
n_layers  = getattr(cfg, 'num_hidden_layers', '???')
n_experts = getattr(cfg, 'num_experts', None) or getattr(cfg, 'num_local_experts', '???')
top_k     = getattr(cfg, 'num_experts_per_tok', None) or getattr(cfg, 'top_k', '???')

print("=" * 65)
print(f"{'PROPERTY':<35} {'PHI-3.5-MoE':>12} {'OLMoE-1B-7B':>12}")
print("=" * 65)
print(f"{'Num layers':<35} {'32':>12} {str(n_layers):>12}")
print(f"{'Experts per layer':<35} {'16':>12} {str(n_experts):>12}")
print(f"{'Top-k routing':<35} {'2':>12} {str(top_k):>12}")
print(f"{'model_dim (LoRA dimension)':<35} {'4096':>12} {str(model_dim):>12}")
print(f"{'ffn_dim (expert internal)':<35} {'6400':>12} {str(ffn_dim):>12}")
print(f"{'Random Jaccard baseline':<35} {'2/16=12.5%':>12} {f'{top_k}/{n_experts}={int(top_k)/int(n_experts)*100:.1f}%' if isinstance(top_k, int) and isinstance(n_experts, int) else '???':>12}")
print("=" * 65)
print()
print("FILL IN MANUALLY FROM CELLS ABOVE:")
print(f"  Experts packed (like Phi) or separate modules? → [ ]")
print(f"  Calling convention (experts(h, ids, wts) or loop)? → [ ]")
print(f"  output_router_logits=True supported? → [ ]")
print(f"  router_logits shape: (B*S, E) or (B, S, E)? → [ ]")
print(f"  Accelerate hook present? → [ ]")
print()
print("WHAT NEEDS TO CHANGE IN HierarchicalPhimoeExperts:")
print("  [ ] Rename class to HierarchicalOLMoEExperts")
print("  [ ] Update forward() to match OLMoE calling convention")
print("  [ ] Update dimension unpacking (packed vs. separate)")
print("  [ ] Update Jaccard threshold (see Task 2 notebook)")
print("  [ ] Confirm hook transfer approach")

PROPERTY                             PHI-3.5-MoE  OLMoE-1B-7B
Num layers                                    32           16
Experts per layer                             16           64
Top-k routing                                  2            8
model_dim (LoRA dimension)                  4096         2048
ffn_dim (expert internal)                   6400         1024
Random Jaccard baseline               2/16=12.5%   8/64=12.5%

FILL IN MANUALLY FROM CELLS ABOVE:
  Experts packed (like Phi) or separate modules? → [ ]
  Calling convention (experts(h, ids, wts) or loop)? → [ ]
  output_router_logits=True supported? → [ ]
  router_logits shape: (B*S, E) or (B, S, E)? → [ ]
  Accelerate hook present? → [ ]

WHAT NEEDS TO CHANGE IN HierarchicalPhimoeExperts:
  [ ] Rename class to HierarchicalOLMoEExperts
  [ ] Update forward() to match OLMoE calling convention
  [ ] Update dimension unpacking (packed vs. separate)
  [ ] Update Jaccard threshold (see Task 2 notebook)
  [ ] Confirm hook tra